# 🤖 Week 3 — AI APIs & LangChain Intro
## AI Pioneers — Cohort 1 | Phase 2: APIs & Tools

**Mayank Chugh** | Azure OpenAI & GenAI Architecture | 2026  
📺 [YouTube](http://www.youtube.com/@itaienthusiast) · 💻 [GitHub](https://github.com/mayankchugh-learning) · 💼 [LinkedIn](https://www.linkedin.com/in/mchugh77)

---

### 🎯 This week's goal
Build a **multi-turn chatbot with conversation memory** — the same architecture used in enterprise AI assistants.

### ⏱️ Session Plan

| # | Topic | 
|---|-------|
| 1 | Working with AI APIs (auth, request/response, streaming) |
| 2 | Multi-turn conversations — managing chat history | 
| 3 | LangChain fundamentals — why it exists | 
| 4 | Build: memory chatbot with LangChain | 
| 5 | Lab: customise for your domain |

---
> **📌 IT Pro Analogy — this whole week:**  
> An AI API is just a REST endpoint. You POST a JSON payload. You get a JSON response.  
> The only difference is the payload contains human language and the response is AI-generated text.  
> Everything you know about API integration applies here.

---
## 📦 Section 0 — Install Dependencies

In [5]:
print("ok")

ok


In [6]:
# Install all packages for Week 3
# langchain         -> orchestration framework
# langchain-ollama  -> Ollama integration for LangChain
# langchain-core    -> base classes (prompts, memory, runnables)
# ollama            -> local model client

%pip install -q langchain langchain-ollama langchain-core ollama

print("✅ All packages installed")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed


---
## 🔑 Section 1 — Working with Local Ollama APIs
### ⏱️ Target: 25 minutes

**IT Pro Analogy:**
> Calling Ollama is still an API integration pattern you already know.  
> You send a model name + JSON payload to a local endpoint and parse the JSON response.  
> The Python client is a thin wrapper around local HTTP calls.

### The request structure
```
POST http://localhost:11434/api/chat
Body:
  model, messages[], stream
```

In [7]:
import os
import ollama

# Ollama runs locally, so no API key is required.
# Ensure Ollama desktop/service is running first.
client = ollama.Client(host=os.environ.get("OLLAMA_HOST", "http://localhost:11434"))

MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1:8b")
print(f"✅ Ollama client ready — using model: {MODEL}")
print("   Tip: run 'ollama pull llama3.1:8b' once if the model is missing")

✅ Ollama client ready — using model: llama3.1:8b
   Tip: run 'ollama pull llama3.1:8b' once if the model is missing


In [8]:
# -- BASIC API CALL -----------------------------------------------------------
# This is the raw local API call - no LangChain yet.

response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "In exactly 2 sentences, explain what local LLM inference means to a senior network engineer."
        }
    ],
    stream=False,
)

# Inspect the response object
print("📦 Full response object:")
print(f"  model:     {response.get('model')}")
print(f"  done:      {response.get('done')}")
print(f"  created:   {response.get('created_at')}")
print()
print("🤖 Answer:")
print(response["message"]["content"])

📦 Full response object:
  model:     llama3.1:8b
  done:      True
  created:   2026-05-28T17:02:32.615603165Z

🤖 Answer:
Local Large Language Model (LLM) inference refers to the process of running a pre-trained language model on a device or within an application, rather than relying on a cloud-based service for processing and generating text responses. This approach can reduce latency, improve security, and increase reliability by moving the computation closer to where it's needed, while still leveraging the power of large language models.


In [9]:
# -- SYSTEM PROMPT + USER MESSAGE --------------------------------------------
# System prompt = stable behavior rules
# User message  = the changing question

response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": """You are a senior SRE with 20 years of enterprise IT experience.
You respond in bullet points only.
You always end with: RISK LEVEL: LOW / MEDIUM / HIGH / CRITICAL.
You are direct and use no filler words.""",
        },
        {
            "role": "user",
            "content": "Our primary database CPU has been at 95% for 10 minutes. What do I check first?",
        },
    ],
    stream=False,
)

print("🔧 SRE Assistant:")
print("-" * 50)
print(response["message"]["content"])

🔧 SRE Assistant:
--------------------------------------------------
• Current database queries running, including any long-running or blocking transactions
• Recent system logs for errors or warnings related to database performance
• Database configuration and settings, specifically concerning CPU-intensive operations like indexing or optimization
• Top SQL consuming resources, if available in monitoring tools or query analysis software
RISK LEVEL: MEDIUM


In [10]:
# -- STREAMING RESPONSES ------------------------------------------------------
# Streaming = receive chunks as they are generated.

print("📡 Streaming response (tokens arrive as generated):")
print("-" * 50)

stream = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "List 5 common causes of database connection pool exhaustion in plain English.",
        }
    ],
    stream=True,
)

for chunk in stream:
    content = chunk.get("message", {}).get("content", "")
    if content:
        print(content, end="", flush=True)

print()
print("-" * 50)
print("✅ Stream complete")

📡 Streaming response (tokens arrive as generated):
--------------------------------------------------
Here are 5 common causes of database connection pool exhaustion:

1. **Incorrect Pool Configuration**: If the database connection pool is not properly configured, it can lead to a shortage of available connections. This can happen if the initial pool size is too small or if the maximum pool size is set too low.
2. **Database Deadlocks**: When multiple processes are waiting for each other to release locks on database resources, deadlocks can occur. If these deadlocks are not properly handled by the application, they can tie up connections in the pool and prevent new requests from being serviced.
3. **Long-Running Transactions**: If transactions take too long to complete, it can leave connections idle in the pool for an extended period, preventing them from being reused and eventually exhausting the pool.
4. **Insufficient Pool Maintenance**: Failing to regularly clear out stale or broke

### ✅ Section 1 Checkpoint

| Concept | What it is | When to use |
|---------|-----------|-------------|
| `client.chat(..., stream=False)` | Standard blocking call | Short responses, simple integrations |
| `role="system"` message | Stable rules for the session | Persona, format, constraints |
| `client.chat(..., stream=True)` | Chunk-by-chunk streaming | Long responses, real-time UIs |
| local Ollama response metadata | Debug + performance checks | Basic observability |

**🧪 Exercise:** Change the `system` prompt above to make the model respond as a non-technical IT manager. Run and compare.

---
## 💬 Section 2 — Multi-Turn Conversations
### ⏱️ Target: 15 minutes

**What we're doing:** Building conversations that remember previous exchanges.

**IT Pro Analogy:**
> A multi-turn conversation is like a stateful session vs a stateless REST call.  
> Single API call = stateless (no memory).  
> Multi-turn = stateful session (context accumulates in the messages list).  
> You maintain the state — the API is always stateless. You send the full history every time.

In [11]:
# -- MANUAL CONVERSATION HISTORY ----------------------------------------------
# The API has NO memory between calls.
# YOU must send the full conversation history in the messages list every time.

conversation_history = []


def chat(user_message, system_prompt=None):
    """Send a message and get a reply. Automatically maintains history."""
    if system_prompt and not any(msg["role"] == "system" for msg in conversation_history):
        conversation_history.append({"role": "system", "content": system_prompt})

    conversation_history.append({"role": "user", "content": user_message})

    response = client.chat(
        model=MODEL,
        messages=conversation_history,
        stream=False,
    )
    assistant_reply = response["message"]["content"]

    conversation_history.append({"role": "assistant", "content": assistant_reply})
    return assistant_reply


# -- HAVE A 3-TURN CONVERSATION ------------------------------------------------
system = "You are a concise IT support assistant. Keep answers to 2-3 sentences."

print("Turn 1:")
reply1 = chat("We run Windows Server 2019 and Azure. What patch cadence do you recommend?", system)
print(f"  AI: {reply1}\n")

print("Turn 2:")
reply2 = chat("What OS did I just mention?")
print(f"  AI: {reply2}\n")

print("Turn 3:")
reply3 = chat("What is the biggest risk if we skip patching for 3 months?")
print(f"  AI: {reply3}\n")

print(f"📊 History length: {len(conversation_history)} messages")

Turn 1:
  AI: For Windows Server 2019, I recommend deploying cumulative updates (CUs) as soon as they're available on Patch Tuesday, with a maximum of two weeks' delay to allow for testing in a non-production environment. For Azure, ensure your virtual machines are configured to automatically install and reboot during scheduled maintenance windows. This balance between patching and minimizing downtime will help maintain security while avoiding unnecessary disruptions.

Turn 2:
  AI: You mentioned Windows Server 2019.

Turn 3:
  AI: The biggest risk of skipping patches for 3 months on a server like Windows Server 2019 is exposure to known vulnerabilities that could be exploited by attackers, potentially leading to system compromise and data loss. In particular, missing critical or security-only updates can leave you vulnerable to publicly disclosed exploits.

📊 History length: 7 messages


In [12]:
# -- CONTEXT WINDOW MANAGEMENT -------------------------------------------------
# The context window = how much history the model can see at once.

MAX_HISTORY_TURNS = 10  # keep last 10 turns


def chat_with_trim(user_message, system_prompt=None):
    """Chat function that trims history when it gets too long."""
    if system_prompt and not any(msg["role"] == "system" for msg in conversation_history):
        conversation_history.append({"role": "system", "content": system_prompt})

    conversation_history.append({"role": "user", "content": user_message})

    # Preserve optional system prompt at index 0, trim remaining conversation turns.
    has_system = bool(conversation_history and conversation_history[0]["role"] == "system")
    allowed_messages = MAX_HISTORY_TURNS * 2 + (1 if has_system else 0)
    if len(conversation_history) > allowed_messages:
        if has_system:
            trimmed = len(conversation_history) - allowed_messages
            del conversation_history[1 : 1 + trimmed]
        else:
            trimmed = len(conversation_history) - allowed_messages
            del conversation_history[:trimmed]
        print(f"  ✂️  Trimmed {trimmed} old messages from history")

    response = client.chat(model=MODEL, messages=conversation_history, stream=False)
    reply = response["message"]["content"]
    conversation_history.append({"role": "assistant", "content": reply})
    return reply


print("✅ chat_with_trim() ready — handles long conversations gracefully")
print(f"   Max history: {MAX_HISTORY_TURNS} turns")

✅ chat_with_trim() ready — handles long conversations gracefully
   Max history: 10 turns


---
## 🔗 Section 3 — LangChain Fundamentals
### ⏱️ Target: 10 minutes

**What is LangChain and why use it?**

| Without LangChain | With LangChain |
|-------------------|----------------|
| Manual history management | Built-in memory classes |
| Manual prompt building | Prompt templates with variables |
| Manual model switching | Swap models in one line |
| Manual chain wiring | Composable pipes: `prompt | model | parser` |
| Custom retry logic | Built-in |

**IT Pro Analogy:**
> LangChain is to AI APIs what Terraform is to cloud resources.  
> You could write raw API calls by hand — or you could use an abstraction layer  
> that handles the boilerplate, retry logic, and composition for you.

**The LangChain pipe operator: `|`**
```python
chain = prompt | llm | output_parser
# Read as: prompt → feeds into → llm → feeds into → parser
# IT analogy: like a Unix pipe: cat file | grep ERROR | wc -l
```

In [13]:
# -- LANGCHAIN BASICS — PROMPT TEMPLATE + MODEL ------------------------------
# ChatPromptTemplate: a reusable prompt with named variables
# ChatOllama: LangChain wrapper around local Ollama models

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model=MODEL,
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise and practical."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke(
    {
        "domain": "Windows Server administration and Azure infrastructure",
        "question": "What are the top 3 things to check when a server's CPU spikes unexpectedly?",
    }
)

print("🤖 LangChain chain result:")
print("-" * 50)
print(result)

🤖 LangChain chain result:
--------------------------------------------------
When a server's CPU spikes unexpectedly, here are the top 3 things to check:

1. **Resource-intensive processes**: Use Task Manager or Resource Monitor to identify which process is consuming excessive CPU resources. Check for any malware, viruses, or unexpected services running.
2. **Disk I/O and storage**: Verify that disk space is not being consumed rapidly due to log files, temporary files, or other storage-related issues. Check Event Viewer logs for errors related to disk performance.
3. **Resource-hungry applications or services**: Review the server's configuration and application settings to ensure they are optimized for performance. Check for any unnecessary services running in the background, and verify that resource-intensive applications (e.g., databases) are properly configured.

These checks will help you quickly identify the root cause of the CPU spike and take corrective action to prevent future 

In [14]:
# -- THE POWER OF LANGCHAIN: SWAP MODELS IN ONE LINE --------------------------
# This is why LangChain exists - model-agnostic code.
# Change from Ollama to OpenAI/Anthropic without rewriting your chain.

# Current Ollama local:
# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3.1:8b", temperature=0.2)

# To switch to OpenAI (uncomment):
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

# To switch to Anthropic (uncomment):
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-5-sonnet-latest", temperature=0.2)

# The chain stays exactly the same:
# chain = prompt | llm | StrOutputParser()

print("💡 Key insight:")
print("   chain = prompt | llm | StrOutputParser()")
print("   Change ONLY the 'llm' variable to switch AI providers.")
print("   Prompt, chain, and all downstream code stays identical.")

💡 Key insight:
   chain = prompt | llm | StrOutputParser()
   Change ONLY the 'llm' variable to switch AI providers.
   Prompt, chain, and all downstream code stays identical.


---
## 🏗️ Section 4 — Build: Multi-Turn Memory Chatbot
### ⏱️ Target: 25 minutes

**What we're building:**
A chatbot that remembers the full conversation using LangChain's `RunnableWithMessageHistory`.

**IT Pro Analogy:**
> `RunnableWithMessageHistory` is like session state middleware.  
> It intercepts every request, loads the session history, injects it into the prompt,  
> and saves the response back to the session store.  
> Your code just calls `.invoke()` - the middleware handles state.

**Architecture:**
```
User input
    |
    v
RunnableWithMessageHistory  <- loads history for session_id
    |
    v
ChatPromptTemplate  <- injects history into prompt
    |
    v
ChatOllama (LLM)
    |
    v
Response  -> saved back to history store
```

In [15]:
# ── FULL MEMORY CHATBOT WITH LANGCHAIN ─────────────────────────────────────

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# ── STEP 1: Prompt with history placeholder ───────────────────────────────
# MessagesPlaceholder("history") = where past messages get injected
# IT analogy: like a session variable slot in ASP.NET or Spring

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant specialised in {domain}.
Keep answers concise — 2–4 sentences unless the user asks for detail.
Use plain English, not jargon."""),
    MessagesPlaceholder("history"),   # ← past messages injected here
    ("human", "{input}"),
])

# ── STEP 2: Build the base chain ──────────────────────────────────────────
chain = prompt | llm

# ── STEP 3: In-memory session store ──────────────────────────────────────
# Keys = session IDs (like user IDs or chat IDs)
# Values = InMemoryChatMessageHistory objects
# IT analogy: like a dictionary of session objects in a web server
_session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Retrieve or create a session history for the given session ID."""
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

# ── STEP 4: Wrap chain with history management ────────────────────────────
# RunnableWithMessageHistory intercepts every .invoke() call:
#   1. Loads history for session_id
#   2. Injects into prompt via MessagesPlaceholder
#   3. Calls the chain
#   4. Saves the response back to the session store

chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",      # which key in invoke() dict is the user message
    history_messages_key="history",  # which MessagesPlaceholder to populate
)

print("✅ Memory chatbot built successfully")
print("   Supports: unlimited concurrent sessions, automatic history management")

✅ Memory chatbot built successfully
   Supports: unlimited concurrent sessions, automatic history management


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3548: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
# ── RUN THE CHATBOT — MULTI-TURN CONVERSATION ─────────────────────────────

# Configuration: which session and domain to use
SESSION_ID = "it-ops-session-1"
DOMAIN     = "internal IT operations — Windows Server, Azure, and network infrastructure"

def ask(user_input: str) -> str:
    """Send a message to the chatbot and return the response."""
    response = chatbot.invoke(
        {"input": user_input, "domain": DOMAIN},
        config={"configurable": {"session_id": SESSION_ID}}
    )
    return response.content


# ── 6-TURN CONVERSATION (watch it remember context) ───────────────────────
turns = [
    "We run Windows Server 2019 on-prem and Azure VMs. What patch cadence do you recommend?",
    "What OS and cloud did I just mention?",  # tests memory
    "What is the biggest risk if we skip patching for 3 months?",
    "Summarise the advice you've given me so far in 3 bullet points.",  # tests full recall
    "What tools would you use to automate patching across both environments?",
    "Which of those tools is easiest to set up for a team of 2 IT engineers?",
]

for i, turn in enumerate(turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}: {turn}")
    print("-"*60)
    print(ask(turn))

# Show how many messages are in memory
history = get_session_history(SESSION_ID)
print(f"\n📊 Session history: {len(history.messages)} messages stored")


Turn 1: We run Windows Server 2019 on-prem and Azure VMs. What patch cadence do you recommend?
------------------------------------------------------------
For a hybrid environment like yours, I'd recommend a monthly patch cadence for both on-prem and Azure VMs. This allows you to stay up-to-date with the latest security patches and features while minimizing downtime.

Consider implementing a "Patch Tuesday" schedule, where you apply all available updates on the second Tuesday of each month. This way, you'll be aligned with Microsoft's regular update cycle and can plan for maintenance windows accordingly.

Turn 2: What OS and cloud did I just mention?
------------------------------------------------------------
You mentioned Windows Server 2019 (on-prem) and Azure VMs.

Turn 3: What is the biggest risk if we skip patching for 3 months?
------------------------------------------------------------
If you skip patching for 3 months, the biggest risk is a potential zero-day exploit being 

In [17]:
# ── MULTIPLE SESSIONS — DIFFERENT USERS, ISOLATED HISTORIES ──────────────
# Each session_id gets its own isolated conversation history.
# IT analogy: like separate user sessions in a web app — different cookies,
# different session state, zero cross-contamination.

def ask_as(session_id: str, domain: str, user_input: str) -> str:
    """Chat as a specific user (session) on a specific domain."""
    response = chatbot.invoke(
        {"input": user_input, "domain": domain},
        config={"configurable": {"session_id": session_id}}
    )
    return response.content

# User A — network engineer talking about firewall
print("👤 USER A:")
ask_as("user-A", "enterprise network security",
        "I manage 50 Cisco firewalls. What log retention policy do you recommend?")
reply_a = ask_as("user-A", "enterprise network security", "What devices did I mention?")
print(f"  A's chatbot remembers: {reply_a[:100]}...")

print()

# User B — DBA with completely separate conversation
print("👤 USER B:")
ask_as("user-B", "SQL Server database administration",
        "I manage 12 SQL Server instances. How should I structure my backup jobs?")
reply_b = ask_as("user-B", "SQL Server database administration", "What databases did I mention?")
print(f"  B's chatbot remembers: {reply_b[:100]}...")

print()
print(f"📊 Total active sessions: {len(_session_store)}")
for sid, hist in _session_store.items():
    print(f"   {sid}: {len(hist.messages)} messages")

👤 USER A:
  A's chatbot remembers: You mentioned Cisco firewalls....

👤 USER B:
  B's chatbot remembers: You mentioned 12 SQL Server instances, but not specific databases....

📊 Total active sessions: 3
   it-ops-session-1: 12 messages
   user-A: 4 messages
   user-B: 4 messages


---
## 🧪 Section 5 — Lab: Customise for Your Domain
### ⏱️ Target: 15 minutes

**Your task:** Change the chatbot to work for YOUR real work domain.

This is the cell you'll personalise. Steps:
1. Change `MY_DOMAIN` to your actual area of work
2. Change `MY_SYSTEM_PROMPT` to set rules relevant to your domain
3. Write 6+ turns based on real questions you face at work
4. Observe how the chatbot remembers and builds on previous answers

In [18]:
# ── YOUR PERSONALISED CHATBOT ─────────────────────────────────────────────
# Change these to match YOUR work domain

MY_DOMAIN = "TODO: paste your work domain here (e.g. 'Azure cloud infrastructure and DevOps')"

MY_SYSTEM_PROMPT = """You are a helpful assistant specialised in {domain}.
Keep answers practical and concise.
TODO: Add 2-3 rules specific to your domain
e.g. 'Always mention cost implications for cloud resources'
     'Flag any security risks in your recommendations'
     'Assume team size of 5 engineers unless told otherwise'"""

# Rebuild the chatbot with your custom system prompt
my_prompt = ChatPromptTemplate.from_messages([
    ("system", MY_SYSTEM_PROMPT),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
my_chain = my_prompt | llm
my_chatbot = RunnableWithMessageHistory(
    my_chain, get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def my_ask(user_input: str) -> str:
    response = my_chatbot.invoke(
        {"input": user_input, "domain": MY_DOMAIN},
        config={"configurable": {"session_id": "my-session"}}
    )
    return response.content

# ── YOUR 6-TURN CONVERSATION ───────────────────────────────────────────────
# Replace these TODOs with real questions from your work
my_turns = [
    "TODO: Ask your first work-related question here",
    "TODO: Follow-up question that builds on the first answer",
    "TODO: Ask what you mentioned earlier (tests memory)",
    "TODO: Ask for a deeper dive on one specific point",
    "TODO: Ask for a summary of everything discussed so far",
    "TODO: Ask for the single most important action to take",
]

for i, turn in enumerate(my_turns, 1):
    if turn.startswith("TODO"):
        print(f"Turn {i}: [Not filled in yet — replace the TODO above]")
        continue
    print(f"\n{'='*55}")
    print(f"Turn {i}: {turn}")
    print("-"*55)
    print(my_ask(turn))

Turn 1: [Not filled in yet — replace the TODO above]
Turn 2: [Not filled in yet — replace the TODO above]
Turn 3: [Not filled in yet — replace the TODO above]
Turn 4: [Not filled in yet — replace the TODO above]
Turn 5: [Not filled in yet — replace the TODO above]
Turn 6: [Not filled in yet — replace the TODO above]


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3548: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


---
## 🎯 Section 6 — Key Concepts Reference

| Concept | IT Analogy | Code |
|---------|-----------|------|
| Local model service | Internal service endpoint | `ollama.Client(host=...)` |
| `messages` list | Session log / audit trail | `[{role, content}, ...]` |
| system message | Connection config / session defaults | `{"role": "system", ...}` |
| Streaming | `tail -f` on a log file | `client.chat(..., stream=True)` |
| Context window | RAM limit | model context + history length |
| LangChain pipe `|` | Unix pipe | `prompt | llm | parser` |
| `MessagesPlaceholder` | Session state slot | Injected into prompt at runtime |
| `session_id` | Cookie / session token | Isolates user conversations |
| `RunnableWithMessageHistory` | Session middleware | Auto load/save per session |
| `InMemoryChatMessageHistory` | In-memory session store | Dict keyed by session_id |

---
## 📋 Deliverable Checklist

- [ ] Notebook runs top-to-bottom without errors
- [ ] You changed `MY_DOMAIN` to your actual work area
- [ ] You wrote 6+ real conversation turns
- [ ] The chatbot correctly remembers context from turn 1 in turn 3+
- [ ] **Stretch:** Add a `clear_history()` function that resets the conversation
- [ ] **Stretch:** Add a second session for a different role or domain and compare responses

---
### 📬 Connect with Mayank Chugh
- 📺 [YouTube](http://www.youtube.com/@itaienthusiast)
- 💻 [GitHub](https://github.com/mayankchugh-learning)
- 💼 [LinkedIn](https://www.linkedin.com/in/mchugh77)
- 📅 [Book a Call](https://topmate.io/mayank_chugh)
- 🛒 [Course](https://mayanklearns.gumroad.com/l/aaskky)